<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_4_1_MLR_Ames_Part1_Revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLR Predicting Housing Prices in Ames Iowa: Part 1
## Data Cleaning

**Data Source:** http://jse.amstat.org/v19n3/decock/AmesHousing.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Data source
url = 'https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_housing_ames.txt'

# Loading the dataframe
df_raw = pd.read_csv(url, sep='\t')

# Initial cleaning: remove extreme outliers per the author's recommendation, large houses sold for little money due to inheritance
df = df_raw.loc[df_raw['Gr Liv Area'] < 4000, :].copy()

# helper functions
def safe_drop(df: pd.DataFrame, drop_list: list[str]) -> pd.DataFrame:
  """
  Safely drops columns from a Pandas DataFrame if they exist.

  Args:
    df (pd.DataFrame): The input DataFrame.
    drop_list (list[str]): A list of column names to attempt to drop.

  Returns:
    pd.DataFrame: The DataFrame with specified columns dropped (if they existed).
  """
  # Filter drop_list to only include columns that exist in the DataFrame
  existing_cols_to_drop = [col for col in drop_list if col in df.columns]

  # Drop the columns if there are any to drop
  if existing_cols_to_drop:
    df = df.drop(existing_cols_to_drop, axis=1)
  return df


## Examine the Dataframe
Use `.info()` to examine your dataframe. Look closely at the "Non-Null Count."


In [ ]:
df.info()

In [ ]:
df.head()

## Problem 2: Remove Obviously Uninformative Features

Order and PID are unique identifiers and should not be used in our model.

In [ ]:
print(df.shape)
df = safe_drop(df, ['Order', 'PID'])
print(df.shape)


Let's look for columns that are monotonic.

In [ ]:
monotonic_list = [x for x in df.columns if len(df[x].unique()) <= 1]
monotonic_list

Look for duplicate rows and rows where all values are missing.

In [ ]:
print(f"Number of duplicate rows: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Number of rows after dropping duplicates: {df.shape[0]}\n")

print(f"Number of rows with all NaNs: {df.isna().all(axis=1).sum()}")
df = df.dropna(how='all')
print(f"Number of rows after dropping all-NaN rows: {df.shape[0]}")

In [ ]:
df.head()

## Problem 4: Data Cleaning - Yolked Variables

We are looking for "yolked features" where the value of one feature is dependent upon the value of another. The most common type of yolked feature here is where a house feature is missing (coded as None, hopefully) and the numeric values describing that missing feature is zero.

The code to automate the search for these features is not included in this notebook. You do not need to look at it or know how it works. The analysis identified 38 "yolked feature" pairs, where the 'None' category in a one-hot encoded categorical feature consistently corresponds to a 0 value in a related numerical feature.
*   **Masonry Veneer:** When 'Mas Vnr Type' is 'None', 'Mas Vnr Area' is consistently 0.
*   **Basement Features:** All basement-related categorical features (`Bsmt Qual`, `Bsmt Cond`, `Bsmt Exposure`, `BsmtFin Type 1`, `BsmtFin Type 2`) are yolked with all basement area numerical features (`BsmtFinSF1`, `BsmtFinSF2`, `Bsmt Unf SF`, `Total Bsmt SF`). This means if any of these categorical basement features is 'None' (indicating no basement), then all basement area measurements are 0.
*   **Garage Features:** All garage-related categorical features (`Garage Type`, `Garage Finish`, `Garage Qual`, `Garage Cond`) are yolked with numerical features describing the garage (`Garage Yr Blt`, `Garage Cars`, `Garage Area`). This implies that if a house has no garage (categorical 'None'), then its garage-related numerical attributes are 0.
*   **Fireplace Features:** 'Fireplace Qu' is yolked with 'Fireplaces'.
*   **Misc Features:** 'Misc Feature' is yolked with 'Misc Val'.
*   **Pool Features:** 'Pool QC' is yolked with 'Pool Area'.

*   **Other Yolked Pairs:** These have probably been erroneously idenfied by our code.
    *   'Electrical' is yolked with 'Garage Area'.
    *   'Fence' is yolked with 'Wood Deck SF'.
    

### Resolving the Yolked Variable Issue
One way to think about this problem is that we have information shared across two features, and what we'd like to do is to have each feature be relatively independent of one another.

In [ ]:
print(df.shape)

### Pool and Misc Features:
# Drop these entirely (both categorical and numerical).
# 99% of houses don't have pools or "misc" features.
# Keeping them adds noise with almost no predictive value for a beginner model.

df = safe_drop(df, ['Pool QC', 'Pool Area', 'Misc Feature', 'Misc Val'])

### Garage Yr Blt:
# If a house has no garage, the year built is often coded as 0 or NA.
# A year of 0 will act as a massive outlier.
# I'd be tempted to do a median replacement, let's drop to keep things simpler for a beginner model.
df = safe_drop(df, ['Garage Yr Blt'])

### Other drops:
# To try to make this exercise shorter, I'm not going to explain these.
more_drops_list = ['Alley', 'Fence', 'Mas Vnr Type', 'Bsmt Qual', 'Bsmt Cond', \
                   'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Fireplace Qu', \
                   'Neighborhood', 'MS Subclass', 'Mo Sold', 'Kitchen Qual', 'Exter Qual', \
                   'Heating QC']
df = safe_drop(df, more_drops_list)

print(df.shape)


In [ ]:
# lets look at garage related features in more detail
garage_list = [col_name for col_name in df.columns if 'garage' in col_name.lower()]

print(df[garage_list].info(), '\n')

# explore garage categoricals
for col in df[garage_list].select_dtypes(include=['object']).columns:
  print(f"{col}: {df[col].value_counts(normalize=True, dropna=False)}\n")

# garage qual and garage cond are 90% a single value, let's drop these
df = safe_drop(df, ['Garage Qual', 'Garage Cond'])

# garage type has some very uncommon values lets collapse into a binary
if 'garage_attached' not in df.columns:
  df['garage_attached'] = np.where(df['Garage Type'] == 'Attchd', 1, 0)

# same for garage unfinished, let's collapse
if 'garage_unfinished' not in df.columns:
  df['garage_unfinished'] = np.where(df['Garage Finish'] == 'Unf', 1, 0)

df = safe_drop(df, ['Garage Type', 'Garage Finish'])



## Cleaning Categoricals

In [ ]:
# let's look for unbalanced categoricals, starting with features where
# one category accounts for more than 70% or the values
for col in df.select_dtypes(include=['object']).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.70:
    print(df[col].value_counts(normalize=True))
    print()

unbalanced_categorical_drop_list = ['Street', 'Land Contour', 'Utilities', \
                                    'Land Slope', 'Condition 1', 'Condition 2',\
                                    'Roof Matl', 'Exter Cond', 'Heating', 'Central Air', \
                                    'Electrical', 'Functional', 'Paved Drive', "Sale Type"]

df = safe_drop(df, unbalanced_categorical_drop_list)

In [ ]:
# let's broaden the search a bit to look at above 50%
for col in df.select_dtypes(include=['object']).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.50:
    print(df[col].value_counts(normalize=True))
    print()
    # create a new feature that binary encodes the top feature versus all others
    top_value = (df[col].value_counts(normalize=True, dropna=False).index[0])
    df[col + '_' + top_value] = np.where(df[col] == top_value, 1, 0)
    df = safe_drop(df, [col])


In [ ]:
# Let's see what remains
for col in df.select_dtypes(include=['object']).columns:
  print(df[col].value_counts(normalize=True))
  print()

# Convert Foundation to Three Categories
df.loc[~df['Foundation'].isin(['PConc', 'CBlock']), 'Foundation'] = 'Other'

# I have no good plan for the exteriors, and I don't want to explode the one_hots
# so lets drop
df = safe_drop(df, ['Exterior 1st', 'Exterior 2nd'])

In [ ]:
# verify foundation and convert to category
for col in df.select_dtypes('object').columns:
  print(col, '\n', df[col].value_counts(dropna=False, normalize=True), '\n')
  df[col] = df[col].astype('category')

## Cleaning Numerics

In [ ]:
#recheck the current status
df.info()

### Identify Binaries and Switch to Bool

In [ ]:
for col in df.columns:
  if df[col].value_counts().shape[0] == 2:
    df[col] = df[col].astype('bool')

df.select_dtypes('boolean').info()

### Unbalanced Numerics

In [ ]:
# look for 'unbalanced' numerics, greater than 90%, drop them
for col in df.select_dtypes(include=np.number).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.90:
    print(df[col].value_counts(normalize=True))
    print()
    df = safe_drop(df, [col])

In [ ]:
# look for 'unbalanced' numerics, greater than 80%, since these are 0 or not 0, swith to bool
for col in df.select_dtypes(include=np.number).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.80:
    print(df[col].value_counts(normalize=True))
    print()
    if len(df[col].unique()) > 2:
      df[col] = np.where(df[col] > 0, 1, 0)
    df[col] = df[col].astype('boolean')

In [ ]:
# look for 'unbalanced' numerics, greater than 59%
for col in df.select_dtypes(include=np.number).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.59:
    print(df[col].value_counts(normalize=True))
    print()

# I'll leave Half Bath in, just because I'd personally want it in the model.

# Mas Vnr Area (Masonry Veneer area) is 60% zero, dropping this one
# Masonry veneer is usually associated with 'wealthy' looking houses
# It would be worth holding onto this for non-basic models.
df = safe_drop(df, ['Mas Vnr Area'])

In [ ]:
# recheck missing values
df.info()

In [ ]:
# median replacement for all remaining numeric types
num_cols_with_na = df.select_dtypes(include=np.number).columns[df.select_dtypes(include=np.number).isnull().any()].tolist()
for col in num_cols_with_na:
    df[col] = df[col].fillna(df[col].median())

df.info()

## Conclusion
We'd made (and documented) a bunch of choices in how we cleaned and simplified our data. We've made these choice knowing that we intend to use multiple linear regression techniques. If we planned on using other techniques (like what we will see in a few weeks), we'd leave in the more and clean a bit less.